In [ ]:
import mne
import numpy as np
import os
from glob import glob
from mne_bids import BIDSPath, write_raw_bids

In [ ]:
proj_dir = r"C:\Users\Laura\OneDrive - Northwestern University\SoundBrain Lab - EAM1"
data_dir = os.path.join(proj_dir, 'EEG_raw')

# output BIDS directory to be created/filled
bids_dir = os.path.join(proj_dir,
                        'data-bids')

In [ ]:
# define function to generate event labels and codes
# (differs per participant)
def generate_event_dict(sub_label, task_label):

    sub_label = int(sub_label)

    if task_label == 'motor':
        if sub_label in [2, 5, 6, 7]:
            event_dict = {'button_press': 6144}
        elif sub_label in [3, 4, 9, 11]:
            event_dict = {'button_press': 34816}
        else:
            event_dict = {'button_press': 1}
            
    elif task_label == 'active':
        if sub_label in [2, 5, 6, 7]:
            event_dict = {'active/pos': 6145,
                          'active/neg': 6146}
        elif sub_label in [3, 4, 9, 11]:
            event_dict = {'active/pos': 34817,
                          'active/neg': 34818}
        elif sub_label == 10:
            event_dict = {'active/pos': 2049,
                          'active/neg': 2050}  
        elif sub_label == 12:
            event_dict = {'active/pos': 1,
                          'active/neg': 2}            
        elif sub_label > 19:
            event_dict = {'active/pos': 3,
                          'active/neg': 4}
    else:
        if sub_label < 12:
            event_dict = {'passive/pos': 2049,
                          'passive/neg': 2050}
        else:
            event_dict = {'passive/pos': 1,
                          'passive/neg': 2}

    return event_dict

In [ ]:
# define input EEG file
# bad participants: 01, 08, 13
event_codes = []
for sub_num in range(11, 20):
    if sub_num == 8:
        continue
    for task_label in ['active']:
        for run_label in range(1,7):
            sub_label = f'{sub_num:02d}'

            data_fpath = os.path.join(data_dir,
                                    f'sub-{sub_label}_task-{task_label}_run-{run_label}.bdf')
            
            try:
                # read in input EEG file
                raw_data = mne.io.read_raw_bdf(data_fpath, verbose=False)

                # find events in the `Status` channel of the raw data
                events_raw = mne.find_events(raw_data, 
                                            stim_channel='Status', 
                                            initial_event=True,
                                            verbose=False)
                

                # see what the event are:
                unique_events = np.unique(events_raw[:, 2])

                if len(event_codes) != len(unique_events):
                    event_codes = np.zeros_like(unique_events)
                    event_codes = unique_events

                if any(event_codes != unique_events):
                    event_codes =  unique_events
                
                print(f"{sub_num}")
                for e in unique_events:
                    print(f"event: {e} occurs {(events_raw[:, 2] == e).sum()} times")
                
                if (task_label == 'active') & ((int(sub_label) > 12) & (int(sub_label) < 20)):
                    # fixing the events because the button and polarity 1 are identical (both 1)
                    # add 2 to every 1 event only when it is preceded by another 1 event (so that polarity 1 events are coded as 3)
                    event_col = events_raw[:, 2]
                    mask = (event_col == 1) & (np.roll(event_col, 1) == 1)
                    mask[0] = False  # first row has no previous value, exclude
                    events_raw[mask, 2] += 2
                    event_dict = {'active/pos': 3,
                                  'active/neg': 2}

                else:
                    # run the function to generate events
                    event_dict = generate_event_dict(sub_label, task_label)

                # only keep task-related events
                if task_label == 'active':
                    include_events = [event_dict['active/pos'], event_dict['active/neg']]
                elif task_label == 'motor':
                    include_events = [event_dict['button_press']]
                else:
                    include_events = [event_dict['passive/pos'], event_dict['passive/neg']]

                events = mne.pick_events(events_raw, include=include_events)

                # define the BIDSPath object for converting this EEG file
                bids_path = BIDSPath(subject=str(sub_label), 
                                    run=run_label, 
                                    task=task_label,
                                    datatype='eeg', 
                                    root=bids_dir, )
                
                # finally, do the raw-to-BIDS conversion
                write_raw_bids(raw_data, 
                            bids_path=bids_path,   
                            events=events,
                            event_id=event_dict,
                            overwrite=True
                            )
            except FileNotFoundError:
                print('file does not exist:', data_fpath)
            except Exception as e:
                print(f"Error occurred: {e}")